Vamos a tantear diferentes modelos y configuraciones antes de parametrizar el definitivo. Gracias a la libreria lazypredict hara un tanteo en los principales modelos de regresion y clasificacion.



In [15]:
import pandas as pd
import numpy as np, random
random.seed(42)

In [2]:
df2 = pd.read_csv('C:/Users/Josue/4GA.Datascience/4GA.DataScience/data/processed/presplit/dfmodel.csv')

In [4]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 62664 entries, 0 to 62663
Data columns (total 49 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   cntry                           62664 non-null  object 
 1   pplfair                         62664 non-null  float64
 2   pplhlp                          62664 non-null  float64
 3   ppltrst                         62664 non-null  float64
 4   lrscale                         62664 non-null  float64
 5   polintr                         62664 non-null  float64
 6   stfdem                          62664 non-null  float64
 7   stfeco                          62664 non-null  float64
 8   stfgov                          62664 non-null  float64
 9   trstep                          62664 non-null  float64
 10  trstlgl                         62664 non-null  float64
 11  trstplc                         62664 non-null  float64
 12  trstplt                         

In [5]:
import pandas as pd
maper = {
    'Bajo': '0',
    'Medio': '1',
    'Alto': '2',
}
columnas = ['stfeco3d', 'trstlgl3d']
for columna in columnas:
    df2[columna] = df2[columna].map(maper).astype(int)  # Añadido .astype(int)

df2.info()

ValueError: cannot convert float NaN to integer

In [6]:
df2.to_csv('C:/Users/Josue/4GA.Datascience/4GA.DataScience/data/processed/presplit/dfmodel.csv', index=False)

In [ ]:
import pandas as pd
from sklearn.model_selection import StratifiedShuffleSplit
#Reducimos las muestras en 3k para el entreno.
def df3kstrat(df, num_muestras, columna_estratificacion=None, random_state=42):


    if columna_estratificacion:
        
        sss = StratifiedShuffleSplit(n_splits=1, train_size=num_muestras, random_state=random_state)
        for train_index, _ in sss.split(df, df[columna_estratificacion]):
            df_reducido = df.iloc[train_index]
    else:
        # Reducción aleatoria sin estratificación
        df_reducido = df.sample(n=num_muestras, random_state=random_state)

    return df_reducido

df3k = df3kstrat(df2, 12000, columna_estratificacion='lr4d')


In [21]:
df3k.to_csv('C:/Users/Josue/4GA.Datascience/4GA.DataScience/data/processed/presplit/df3k.csv', index=False)

In [23]:
import pandas as pd
from sklearn.model_selection import train_test_split
import os
dflrscale = df3k.copy().drop(['lrfc3d', 'lr4d'], axis=1)
def split_and_save_data(df, target_column, output_dir, test_size=0.2, random_state=42, filename_prefix=""):
  X = df.drop(columns=target_column)
  y = df[target_column]
  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state)
  train_dir = os.path.join(output_dir, "train")
  test_dir = os.path.join(output_dir, "test")
  os.makedirs(train_dir, exist_ok=True)
  os.makedirs(test_dir, exist_ok=True)
  X_train.to_csv(os.path.join(train_dir, f"{filename_prefix}_X_train.csv"), index=False)
  X_test.to_csv(os.path.join(test_dir, f"{filename_prefix}_X_test.csv"), index=False)
  y_train.to_csv(os.path.join(train_dir, f"{filename_prefix}_y_train.csv"), index=False)
  y_test.to_csv(os.path.join(test_dir, f"{filename_prefix}_y_test.csv"), index=False)
output_dir = 'C:/Users/Josue/4GA.Datascience/4GA.DataScience/models/Split/'
target_column = 'lrscale'

split_and_save_data(dflrscale, target_column, output_dir, filename_prefix="lrscale")

In [24]:

dflr4d= df3k.copy().drop(['lrscale','lrfc3d'], axis=1)
output_dir = 'C:/Users/Josue/4GA.Datascience/4GA.DataScience/models/Split/'
target_column = 'lr4d'
split_and_save_data(dflr4d, target_column, output_dir, filename_prefix="lr4d")

In [26]:

dflrfc3d= df3k.copy().drop(['lrscale','lr4d'], axis=1)
output_dir = 'C:/Users/Josue/4GA.Datascience/4GA.DataScience/models/Split/'
target_column = 'lrfc3d'
split_and_save_data(dflrfc3d, target_column, output_dir, filename_prefix="lrfc3d")

In [35]:
import pandas as pd
import xgboost as xgb
import os
import joblib
from sklearn.metrics import mean_squared_error
import numpy as np

def train_xgboost_with_feature_iteration(train_x_filename, train_y_filename, test_x_filename, test_y_filename, target_column, output_dir):
    """
    Entrena modelos XGBoost iterando sobre el número de características y guarda los nombres de las predictoras.
    """

    try:
        train_x_df = pd.read_csv(train_x_filename)
        train_y_df = pd.read_csv(train_y_filename)
        test_x_df = pd.read_csv(test_x_filename)
        test_y_df = pd.read_csv(test_y_filename)

        # Verificar el número de muestras
        if len(train_x_df) != len(train_y_df):
            raise ValueError(f"Inconsistent number of samples in train_x_df and train_y_df: [{len(train_x_df)}, {len(train_y_df)}]")

        if len(test_x_df) != len(test_y_df):
            raise ValueError(f"Inconsistent number of samples in test_x_df and test_y_df: [{len(test_x_df)}, {len(test_y_df)}]")

        # Separar características y variable objetivo
        X_train = train_x_df
        y_train = train_y_df[target_column]
        X_test = test_x_df
        y_test = test_y_df[target_column]

        best_num_features = None
        best_rmse = float('inf')
        feature_importances = []
        feature_names = []

        for num_features in range(5, len(X_train.columns) + 1, 5):
            X_train_subset = X_train.iloc[:, :num_features]
            X_test_subset = X_test.iloc[:, :num_features]

            model = xgb.XGBRegressor(objective='reg:squarederror', random_state=42)
            model.fit(X_train_subset, y_train)
            y_pred = model.predict(X_test_subset)
            mse = mean_squared_error(y_test, y_pred)
            rmse = np.sqrt(mse)

            if rmse < best_rmse:
                best_rmse = rmse
                best_num_features = num_features
                feature_importances = model.feature_importances_
                feature_names = X_train_subset.columns.tolist() # Se obtienen los nombres de las predictoras

        # Entrenar el modelo final con el mejor número de características
        X_train_final = X_train.iloc[:, :best_num_features]
        X_test_final = X_test.iloc[:, :best_num_features]
        model_final = xgb.XGBRegressor(objective='reg:squarederror', random_state=42)
        model_final.fit(X_train_final, y_train)

        # Guardar el modelo, la mejor configuración y las importancias de las características
        joblib.dump(model_final, os.path.join(output_dir, f"xgboost_{target_column}_model.joblib"))
        with open(os.path.join(output_dir, f"xgboost_{target_column}_config.txt"), "w") as f:
            f.write(f"Mejor número de características: {best_num_features}\n")
            f.write(f"RMSE: {best_rmse}\n")
            f.write(f"Importancia de las características: {feature_importances}\n")
            f.write(f"Nombres de las predictoras: {feature_names}\n") # Se escriben los nombres de las predictoras

        print(f"Modelo para {target_column} entrenado y guardado.")

    except FileNotFoundError as e:
        print(f"Error: Archivo no encontrado - {e}")
    except ValueError as e:
        print(f"Error de valor: {e}")
    except Exception as e:
        print(f"Error inesperado: {e}")


train_x_filename = "C:/Users/Josue/4GA.Datascience/4GA.DataScience/models/Split/train/lrfc3d_X_train.csv"
train_y_filename = "C:/Users/Josue/4GA.Datascience/4GA.DataScience/models/Split/train/lrfc3d_y_train.csv"
test_x_filename = "C:/Users/Josue/4GA.Datascience/4GA.DataScience/models/Split/test/lrfc3d_X_test.csv"
test_y_filename = "C:/Users/Josue/4GA.Datascience/4GA.DataScience/models/Split/test/lrfc3d_y_test.csv"
target_column = "lrfc3d"
output_dir = "C:/Users/Josue/4GA.Datascience/4GA.DataScience/models/Setups/"
train_xgboost_with_feature_iteration(train_x_filename, train_y_filename, test_x_filename, test_y_filename, target_column, output_dir)

Modelo para lrfc3d entrenado y guardado.


Todos las combinaciones probadas estan en '.\4GA.DataScience\models\Allmodelsresumedefault.txt'